# 03_diagnosis_recheck_CHS

- 목적: 02 진단의 결론 중 뒤집힐 수 있는 세 가지를 재검증한다 — A-1 DC 오프셋의 정체(에일리어싱 산물인가), A-2 진동 스케일 동일성, A-3 형식 누수 차단 증명
- 입력: `data/raw/` 원본 2파일(`data_quality.load`), `data/processed/segments.csv`(`segments.build_segments_table`)
- 출력: `figures/03_diagnosis_recheck_CHS/`, `results/03_diagnosis_recheck_CHS/`
- 담당: CHS
- 공통 모듈: `src/preprocess.py`(형식 통일·DC 제거·형식 피처), `src/split.py`(세그먼트 GroupKFold·시간 블록), `src/signal_checks.py`(사인 피팅)


In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
NB = "03_diagnosis_recheck_CHS"  # 파일명(확장자 제외)과 같게
FIG, RES = paths.nb_dirs(NB)
SEED = 42
np.random.seed(SEED)

In [2]:
import data_quality as dq, preprocess as pp, split as sp, signal_checks as sc, segments as sg
from scipy import stats
from sklearn.metrics import roc_auc_score
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False

dfs = dq.load_all()
raw = pp.add_seg_uid(pd.concat([dfs["normal"], dfs["outlier"]], ignore_index=True))
seg = sg.build_segments_table(save=True)            # data/processed/segments.csv 재생성 (620 × 26)
seg_meta = seg.set_index("seg_uid")[["src", "seg_id", "len", "label", "state", "vib_grade", "cur_grade", "vib_rms"]]
print(raw.shape, seg.shape, raw["seg_uid"].nunique())

(20600, 10) (620, 26) 620


## A-1. DC 오프셋의 정체 — 에일리어싱 산물인가, 실제 오프셋인가

**문제.** 02 §2에서 세그먼트 평균(DC 오프셋) 하나로 정상/이상을 AUC 0.89~0.94로 구분할 수 있어 "파일 형식 누수"로 결론했고, "DC 제거 필수"를 전처리 규칙으로 잡았다. 그런데 전류는 60 Hz 구동이 10 Hz 샘플링에서 접힌 신호다. 구동 주파수가 60 Hz의 정수배 근처에서 조금 벗어나면 접힌 성분은 0.1 Hz 이하의 느린 정현파가 되고, 5초 burst 안에서는 **DC처럼** 보인다. 그렇다면 DC는 형식 누수가 아니라 "구동 주파수 이탈"이라는 물리 신호일 수 있다.

**검증.** 세그먼트마다 `A·sin(2πft+φ)+C`를 최소제곱으로 맞춘다(`signal_checks.sine_fit_segment`, f는 0.01~5 Hz 로그 격자 500점 + 정밀화).
- 가설이 맞으면: 피팅 f가 0.1 Hz 이하로 갈수록 |DC|가 커진다 → 전류 |DC| vs f의 Spearman |rho| ≥ 0.5.
- 이상 세그먼트 16~20(DC 180~272)의 피팅 진폭 A가 정상 수준(약 100~270)이고 f≈0.02~0.1 Hz면 "구동 주파수 이탈", A가 정상보다 작고 C만 크면 "실제 오프셋".

**판정 규칙(게이트 G1).** |rho| ≥ 0.5 → 전처리를 사인 피팅 잔차 기반으로 변경(G1-a). 아니면 세그먼트 평균 제거 유지(G1-b).

*주의.* f가 세그먼트 길이의 역수의 절반보다 작으면(반 주기도 안 들어감) 사인은 선형 추세와 구분되지 않아 A·C가 발산한다. 그런 피팅은 `trend_like=True`로 표시하고 상관 계산에서 제외한다(길이 10샘플 미만도 제외).

In [3]:
raw_u = pp.unify_format(raw)                         # 형식 통일만 (DC는 아직 제거 안 함)
fit = sc.sine_fit_table(raw_u, fs=10.0)               # 620 세그먼트 × 3채널, 약 20~30초
fit = fit.join(seg_meta, on="seg_uid")
fit["dc_abs"] = fit["dc_mean"].abs()
fit["dur_s"] = fit["n"] / 10.0
fit["trend_like"] = fit["f_hz"] < 0.5 / fit["dur_s"]   # 반 주기 미만 → 선형 추세와 구분 불가
fit.to_csv(RES / "sine_fit_by_segment.csv", index=False, encoding="utf-8-sig")
print("trend_like 비율:", fit.groupby(["channel", "src"])["trend_like"].mean().round(3).to_dict())
fit.head()

trend_like 비율: {('AI0_Vibration', 'normal'): 0.017, ('AI0_Vibration', 'outlier'): 0.048, ('AI1_Vibration', 'normal'): 0.007, ('AI1_Vibration', 'outlier'): 0.19, ('AI2_Current', 'normal'): 0.047, ('AI2_Current', 'outlier'): 0.333}


,seg_uid,channel,f_hz,amp,phase,offset,dc_mean,resid_rms,r2,n,src,seg_id,len,label,state,vib_grade,cur_grade,vib_rms,dc_abs,dur_s,trend_like
0,normal_0,AI0_Vibration,3.201612,0.059161,0.993356,-0.007206,-0.006957,0.054664,0.366107,41,normal,0,41,0.0,0,<NA>,<NA>,0.177171,0.006957,4.1,False
1,normal_0,AI1_Vibration,1.789226,0.220483,-2.226875,-0.000776,-0.009436,0.042100,0.933211,41,normal,0,41,0.0,0,<NA>,<NA>,0.177171,0.009436,4.1,False
2,normal_0,AI2_Current,0.600238,210.795079,1.143589,0.721591,19.043902,4.628991,0.999020,41,normal,0,41,0.0,0,<NA>,<NA>,0.177171,19.043902,4.1,False
3,normal_1,AI0_Vibration,3.601012,0.063316,0.833510,0.006001,0.007061,0.057821,0.373719,37,normal,1,37,0.0,0,<NA>,<NA>,0.151397,0.007061,3.7,False
4,normal_1,AI1_Vibration,1.793258,0.177099,0.856815,0.000660,0.006813,0.043472,0.891926,37,normal,1,37,0.0,0,<NA>,<NA>,0.151397,0.006813,3.7,False


In [4]:
# 파일 × 채널 요약: 피팅 f, 진폭, R², 잔차 RMS
ok = fit[(fit["n"] >= 10) & (~fit["trend_like"])]
summ = (ok.groupby(["channel", "src"])
          .agg(n_seg=("seg_uid", "size"), f_med=("f_hz", "median"), f_q25=("f_hz", lambda s: s.quantile(.25)),
               f_q75=("f_hz", lambda s: s.quantile(.75)), amp_med=("amp", "median"), r2_med=("r2", "median"),
               resid_rms_med=("resid_rms", "median"), dc_abs_med=("dc_abs", "median"))
          .round(4))
summ.to_csv(RES / "sine_fit_summary.csv", encoding="utf-8-sig")
summ

n_seg   f_med   f_q25   f_q75   amp_med  r2_med  resid_rms_med  dc_abs_med
channel       src                                                                                
AI0_Vibration normal     522  3.5533  1.7787  3.5983    0.0683  0.5144         0.0431      0.0010
              outlier     16  1.0546  0.8241  1.2021    0.3063  0.3509         0.2951      0.0420
AI1_Vibration normal     529  1.8000  1.7892  3.5716    0.1087  0.7698         0.0413      0.0010
              outlier     13  0.9886  0.6143  1.1345    0.1871  0.3503         0.1827      0.0360
AI2_Current   normal     528  0.6001  0.5987  0.6011  144.6400  0.9965         5.6634      2.8171
              outlier     11  0.4823  0.2081  2.5918   72.0523  0.4635        51.2088     74.3880

In [5]:
# 게이트 G1: 채널별 |DC| vs 피팅 f 의 Spearman 상관 (정상 / 이상 / 합침)
rows = []
for ch in pp.SENSORS:
    s = ok[ok["channel"] == ch]
    for name, g in [("normal", s[s["src"] == "normal"]), ("outlier", s[s["src"] == "outlier"]), ("pooled", s)]:
        rho, p = stats.spearmanr(g["dc_abs"], g["f_hz"])
        rows.append({"channel": ch, "subset": name, "n": len(g), "spearman_rho": round(rho, 3), "p_value": p,
                     "share_f_below_0.1Hz": round((g["f_hz"] < 0.1).mean(), 3)})
gate = pd.DataFrame(rows)
gate.to_csv(RES / "sine_fit_dc_vs_freq_spearman.csv", index=False, encoding="utf-8-sig")
cur_rho = gate.loc[(gate["channel"] == "AI2_Current") & (gate["subset"] == "pooled"), "spearman_rho"].iloc[0]
print("G1 판정:", "G1-a (에일리어싱 산물 → 사인 잔차 전처리)" if abs(cur_rho) >= 0.5 else f"G1-b (|rho|={abs(cur_rho):.3f} < 0.5 → 세그먼트 평균 제거 유지)")
gate

G1 판정: G1-b (|rho|=0.036 < 0.5 → 세그먼트 평균 제거 유지)


,channel,subset,n,spearman_rho,p_value,share_f_below_0.1Hz
0,AI0_Vibration,normal,522,-0.092,0.035005,0.0
1,AI0_Vibration,outlier,16,-0.218,0.418087,0.0
2,AI0_Vibration,pooled,538,-0.153,0.000385,0.0
3,AI1_Vibration,normal,529,-0.093,0.032636,0.0
4,AI1_Vibration,outlier,13,-0.022,0.943186,0.0
5,AI1_Vibration,pooled,542,-0.145,0.000704,0.0
6,AI2_Current,normal,528,-0.043,0.322649,0.0
7,AI2_Current,outlier,11,0.200,0.555445,0.0
8,AI2_Current,pooled,539,-0.036,0.399126,0.0


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, ch in zip(axes, pp.SENSORS):
    s = ok[ok["channel"] == ch]
    for src, c, m in [("normal", "tab:blue", "."), ("outlier", "tab:red", "x")]:
        g = s[s["src"] == src]
        ax.scatter(g["f_hz"], g["dc_abs"], s=14, c=c, marker=m, alpha=.6, label=f"{src} (n={len(g)})")
    r = gate[(gate["channel"] == ch) & (gate["subset"] == "pooled")].iloc[0]
    ax.set_xscale("log"); ax.set_xlabel("fitted f [Hz] (log)"); ax.set_ylabel("|DC| = |segment mean|")
    ax.set_title(f"{ch}: Spearman rho={r['spearman_rho']:.2f}"); ax.axvline(0.1, ls="--", c="gray", lw=.8); ax.legend(fontsize=8)
paths.save_fig(FIG, "sine_fit_freq_vs_dc.png")

Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


saved figures\03_diagnosis_recheck_CHS\sine_fit_freq_vs_dc.png


### 이상 세그먼트 16~20의 정체: 진폭은 작고 오프셋만 크다

In [7]:
cur = fit[fit["channel"] == "AI2_Current"]
normal_amp = cur[(cur["src"] == "normal") & (cur["n"] >= 10) & (~cur["trend_like"])]["amp"]
o = cur[cur["src"] == "outlier"].sort_values("seg_id")[["seg_id", "n", "cur_grade", "f_hz", "amp", "offset", "dc_mean", "resid_rms", "r2", "trend_like"]]
o["amp_pct_of_normal"] = o["amp"].apply(lambda a: round(float((normal_amp < a).mean()), 3))   # 정상 진폭 분포에서의 백분위
o.round(3).to_csv(RES / "sine_fit_outlier_current.csv", index=False, encoding="utf-8-sig")
print("정상 전류 진폭 분위(10/50/90%):", normal_amp.quantile([.1, .5, .9]).round(1).tolist())
o.round(3)

정상 전류 진폭 분위(10/50/90%): [115.3, 144.6, 245.9]


,seg_id,n,cur_grade,f_hz,amp,offset,dc_mean,resid_rms,r2,trend_like,amp_pct_of_normal
1799,0,4,경계,4.999,19635.164,-17.138,-24.725,13.425,0.934,False,1.000
1802,1,50,경계,0.224,72.052,40.575,43.988,73.749,0.309,False,0.000
1805,2,47,경계,0.230,112.819,-16.922,-9.266,60.920,0.641,False,0.006
1808,3,31,경계,4.441,52.681,74.526,73.448,56.070,0.291,False,0.000
1811,4,18,경계,0.219,162.761,-19.339,-120.272,70.786,0.451,True,0.585
1814,5,3,경계,NaN,NaN,-141.833,-141.833,NaN,NaN,False,0.000
1817,6,50,경계,0.181,132.338,-83.593,-74.388,89.930,0.519,False,0.455
1820,7,40,경계,0.106,273.683,-253.406,-81.420,81.742,0.622,True,1.000
1823,8,25,경계,0.010,13943.562,-13903.928,-26.232,72.164,0.374,True,1.000
1826,9,8,경계,3.467,59.333,-114.085,-108.775,33.463,0.572,False,0.000


In [8]:
# 파형 + 피팅 예시: 정상 세그먼트(중앙값 f) vs 이상 17·19 (DC 큰 세그먼트)
def _fitted(row, n):
    t = np.arange(n) / 10.0
    return row["amp"] * np.sin(2 * np.pi * row["f_hz"] * t + row["phase"]) + row["offset"]
ex_normal = cur[(cur["src"] == "normal") & (cur["n"] == 50)].iloc[len(cur[(cur["src"] == "normal") & (cur["n"] == 50)]) // 2]
picks = [("normal", ex_normal["seg_uid"]), ("outlier", "outlier_17"), ("outlier", "outlier_19")]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, (src, uid) in zip(axes, picks):
    x = raw_u.loc[raw_u["seg_uid"] == uid, "AI2_Current"].to_numpy()
    row = cur[cur["seg_uid"] == uid].iloc[0]
    ax.plot(np.arange(len(x)) / 10, x, "k.-", lw=.8, label="AI2_Current")
    ax.plot(np.arange(len(x)) / 10, _fitted(row, len(x)), "r-", lw=1.2, label=f"fit f={row['f_hz']:.3f}Hz A={row['amp']:.0f} C={row['offset']:.0f}")
    ax.axhline(row["dc_mean"], c="tab:blue", ls="--", lw=.8, label=f"mean={row['dc_mean']:.0f}")
    ax.set_title(f"{uid} (R²={row['r2']:.2f}, resid RMS={row['resid_rms']:.1f})", fontsize=9); ax.set_xlabel("s"); ax.legend(fontsize=7)
paths.save_fig(FIG, "sine_fit_examples.png")

saved figures\03_diagnosis_recheck_CHS\sine_fit_examples.png


### 부수 발견: 정상 전류는 0.600 Hz 순수 정현파, 이상은 정현파에서 벗어난다 → 사인 잔차가 피처 후보

In [9]:
# 전류 피팅 부산물의 단독 분리력 (세그먼트 단위 AUC, n>=10). 형식 누수와 무관한 "파형 왜곡" 지표.
c10 = cur[cur["n"] >= 10].copy()
c10["f_dev"] = (c10["f_hz"] - 0.6).abs()
c10["resid_ratio"] = c10["resid_rms"] / c10["amp"].clip(lower=1e-9)
auc_rows = []
for col in ["resid_rms", "r2", "f_dev", "resid_ratio", "dc_abs", "amp"]:
    a = roc_auc_score(c10["label"], c10[col]); auc_rows.append({"feature": col, "auc_sym": round(max(a, 1 - a), 4), "n_normal": int((c10.label == 0).sum()), "n_outlier": int((c10.label == 1).sum())})
auc_tab = pd.DataFrame(auc_rows).sort_values("auc_sym", ascending=False)
auc_tab.to_csv(RES / "sine_fit_feature_auc.csv", index=False, encoding="utf-8-sig")
auc_tab

,feature,auc_sym,n_normal,n_outlier
0,resid_rms,1.0000,530,17
1,r2,1.0000,530,17
2,f_dev,0.9992,530,17
4,dc_abs,0.9203,530,17
3,resid_ratio,0.8824,530,17
5,amp,0.6485,530,17


### A-1 결론 — 게이트 **G1-b**: DC 오프셋은 에일리어싱 산물이 아니다. 세그먼트 평균 제거를 유지한다

- 전류 |DC| vs 피팅 f Spearman rho: 정상 −0.043 (n=528, p=0.35) / 이상 +0.200 (n=11) / 합침 **−0.036** (n=539, p=0.42) → |rho| < 0.5. f가 낮을수록 |DC|가 커지는 관계가 없다. 진동도 −0.153(AI0)·−0.145(AI1)로 약해 진동 DC의 형식 누수 결론은 유지한다.
- 정상 전류는 세그먼트마다 **f = 0.6001 Hz(IQR 0.5987~0.6011), R² 0.9965, 진폭 중앙값 144.6**의 순수 정현파다. 02 §3의 zero-crossing 추정(0.57 Hz)보다 정밀하며, 60 Hz 구동이 10 Hz 샘플링에 접힌 겉보기 주파수가 0.600 Hz라는 뜻이다(실제 구동 주파수 후보 59.4 / 60.6 Hz — `alias_candidates` 참고). 정상 안에서 f 편차가 ±0.001 Hz에 불과하므로 "구동 주파수 이탈 → 느린 정현파 → DC"라는 경로는 정상 데이터에서 발생하지 않는다.
- 이상 세그먼트 17~20의 피팅 진폭은 138 / 99 / 57 / 29로 정상 분포의 47 / 0 / 0 / 0 백분위이고, 오프셋 C는 288 / 237 / 188 / 179다. 즉 **진폭은 줄고 오프셋만 크다** → "실제 오프셋"이다. 파형 그림(`sine_fit_examples.png`)에서 이상 17은 전류가 80~480으로 전부 양수이고 0.6 Hz 성분이 사라져 있다.
- 세그먼트 길이 대비 반 주기 미만인 피팅(`trend_like`, 이상 전류 33%, 정상 5%)은 A·C가 발산해 제외했다. 제외 후 남은 피팅에는 0.1 Hz 미만이 없다(길이 ≤5 s이면 판정 하한이 0.1 Hz이므로 정의상 그렇다).

**부수 발견(5번 창의성 후보).** 정상 전류가 정현파에 극히 잘 맞기 때문에 "정현파에서 얼마나 벗어났는가"가 그 자체로 이상 지표가 된다: 사인 피팅 잔차 RMS(정상 중앙값 5.66 vs 이상 51.2, 약 9배)와 R²(0.9965 vs 0.46)의 세그먼트 단위 AUC가 **1.000**(n=530 / 17), |f − 0.6| 0.999. 이는 1.19 격자 양자화(최대 오차 0.6)로는 만들 수 없는 크기라 형식 누수는 아니다. 다만 이상 이벤트가 1건(하루)뿐이라 "그날 전류 계측 경로가 달랐을 가능성"(날짜 교란)은 배제할 수 없으므로, 이 피처는 진폭 피처와 별도 모델로 두고 A-3 차단 실험과 3번 오류분석에서 재검토한다.

**모델 단계 반영.** `preprocess.remove_dc(method="mean")` 유지(`sine` 미구현). `features.py`에 전류 사인 피팅 부산물(`cur_fit_resid_rms`, `cur_fit_r2`, `cur_fit_f_dev`)을 **별도 피처군**으로 추가하되 기본 모델에는 넣지 않고 ablation으로 비교한다. 02 §3 "FFT·스펙트럼 분석 무의미" 표현은 "결함 주파수 분석 불가, 단 접힌 0.600 Hz 성분의 이탈은 지표 후보"로 정정한다(A-4).

## A-2. 진동 스케일이 두 파일에서 같은가 — 잡음 바닥 비교

**문제.** 이상 파일의 진동 RMS는 정상 대비 최대 6배인데, 이것이 물리 현상인지 계측 게인(스케일) 차이인지 구분되지 않았다. 두 파일의 소수 자릿수(6 vs 8)가 다르다는 것 자체가 서로 다른 저장 경로를 시사한다.

**검증.**
1. `signal_checks.noise_floor`: 1차 차분의 MAD 기반 잡음 표준편차(1.4826·MAD(Δx)/√2). 신호보다 샘플 간 계측 잡음에 민감하다. 두 파일을 같은 자릿수(6자리)로 반올림한 뒤 계산.
2. 정상 **저부하 상태(state1)** 세그먼트와 이상 **조용한 세그먼트 3·19·20**(02 §1에서 진동이 정상 유사) 비교. 게인이 같으면 잡음 바닥이 비슷해야 한다.
3. 값 해상도 표(`value_step_table`): 고유값 수, 최소 값 간격, float32 왕복 오차 → 저장 경로 추정. 전류 1.19209 격자의 정수 몫 범위.
4. DC 제거·반올림 후 진동 샘플 분포의 KS 검정(정상 저부하 vs 이상 조용한 세그먼트).

**판정 규칙(게이트 G2).** 잡음 바닥 비율(이상 조용/정상 저부하)이 0.5~2배 안이면 "스케일 동일"(G2-a). 밖이면 게인 차이 가능성을 한계로 명시하고 A-3에 "세그먼트 내 z-score" 변형을 추가(G2-b).

In [10]:
nf = sc.noise_floor_table(raw, decimals=6, min_len=10).join(seg_meta)
nf["group"] = np.select(
    [(nf["src"] == "normal") & (nf["state"] == 1), (nf["src"] == "normal") & (nf["state"] == 0),
     (nf["src"] == "outlier") & nf["seg_id"].isin([3, 19, 20]), nf["src"] == "outlier"],
    ["normal_state1(저부하)", "normal_state0(고부하)", "outlier_quiet(3·19·20)", "outlier_other"], default="?")
cols = [c for c in nf.columns if c.endswith(("_nf", "_ac_rms", "_lsb"))]
cmp = nf.groupby("group")[cols].median().round(5)
cmp.insert(0, "n_seg", nf.groupby("group").size())
ratio = (cmp.loc["outlier_quiet(3·19·20)", ["AI0_Vibration_nf", "AI1_Vibration_nf"]] / cmp.loc["normal_state1(저부하)", ["AI0_Vibration_nf", "AI1_Vibration_nf"]]).round(3)
cmp.loc["ratio quiet/normal_state1"] = np.nan
cmp.loc["ratio quiet/normal_state1", ["AI0_Vibration_nf", "AI1_Vibration_nf"]] = ratio.values
cmp.to_csv(RES / "noise_floor_compare.csv", encoding="utf-8-sig")
nrm = nf[nf["src"] == "normal"]
rho_nf_rms = {ch: round(stats.spearmanr(nrm[f"{ch}_nf"], nrm[f"{ch}_ac_rms"])[0], 3) for ch in pp.VIB_CHANNELS}
print("잡음 바닥 비율(이상 조용/정상 저부하):", ratio.to_dict())
print("G2 판정:", "G2-a (스케일 동일)" if ratio.between(0.5, 2).all() else "G2-b (0.5~2배 밖 → 게인 차이 가능성, A-3에 z-score 변형 추가)")
print("정상 내 잡음 바닥 vs AC RMS Spearman:", rho_nf_rms, "→ 10 Hz에서는 잡음 바닥이 진폭과 함께 움직인다")
cmp

잡음 바닥 비율(이상 조용/정상 저부하): {'AI0_Vibration_nf': 0.385, 'AI1_Vibration_nf': 0.334}
G2 판정: G2-b (0.5~2배 밖 → 게인 차이 가능성, A-3에 z-score 변형 추가)
정상 내 잡음 바닥 vs AC RMS Spearman: {'AI0_Vibration': np.float64(0.659), 'AI1_Vibration': np.float64(0.912)} → 10 Hz에서는 잡음 바닥이 진폭과 함께 움직인다


,n_seg,AI0_Vibration_nf,AI0_Vibration_ac_rms,AI0_Vibration_lsb,AI1_Vibration_nf,AI1_Vibration_ac_rms,AI1_Vibration_lsb
group,,,,,,,
normal_state0(고부하),226.0,0.07486,0.08259,0.00015,0.14123,0.16909,0.00027
normal_state1(저부하),304.0,0.06140,0.05178,0.00010,0.06002,0.05230,0.00013
outlier_other,14.0,0.07105,0.38105,0.00029,0.04413,0.23676,0.00023
outlier_quiet(3·19·20),3.0,0.02362,0.02627,0.00105,0.02005,0.06412,0.00021
ratio quiet/normal_state1,NaN,0.38500,NaN,NaN,0.33400,NaN,NaN


In [11]:
nf[nf["group"] == "outlier_quiet(3·19·20)"][["seg_id", "n", "vib_grade", "AI0_Vibration_nf", "AI1_Vibration_nf", "AI0_Vibration_ac_rms", "AI1_Vibration_ac_rms"]].round(4)

,seg_id,n,vib_grade,AI0_Vibration_nf,AI1_Vibration_nf,AI0_Vibration_ac_rms,AI1_Vibration_ac_rms
seg_uid,,,,,,,
outlier_3,3,31,정상 유사,0.0236,0.0131,0.0287,0.0643
outlier_19,19,10,정상 유사,0.0045,0.0467,0.0263,0.0641
outlier_20,20,15,정상 유사,0.0243,0.0201,0.0214,0.0210


In [12]:
order = ["normal_state1(저부하)", "normal_state0(고부하)", "outlier_quiet(3·19·20)", "outlier_other"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, ch in zip(axes, pp.VIB_CHANNELS):
    data = [nf.loc[nf["group"] == g, f"{ch}_nf"].dropna() for g in order]
    ax.boxplot(data, tick_labels=[f"{g}\n(n={len(d)})" for g, d in zip(order, data)], showfliers=True)
    for i, d in enumerate(data, 1):
        ax.scatter(np.random.normal(i, .04, len(d)), d, s=8, alpha=.4, c="tab:red" if "outlier" in order[i-1] else "tab:blue")
    ax.set_ylabel("noise floor (diff-MAD)"); ax.set_title(f"{ch}: 잡음 바닥 (6자리 반올림 후)", fontsize=10); ax.tick_params(axis="x", labelsize=7)
paths.save_fig(FIG, "noise_floor_box.png")

saved figures\03_diagnosis_recheck_CHS\noise_floor_box.png


In [13]:
# 값 해상도·저장 경로 추정
vs = sc.value_step_table(dfs)
vs.to_csv(RES / "value_step_table.csv", index=False, encoding="utf-8-sig")
for src in ["normal", "outlier"]:
    q = dfs[src]["AI2_Current"] / pp.CUR_STEP
    frac = np.abs(q - np.round(q))
    print(f"{src}: 전류/1.19209 정수 비율 {(frac < 1e-3).mean():.4f}, 정수 몫 범위 {np.round(q).min():.0f}~{np.round(q).max():.0f}")
print("1.19209 ≈ 10 V / 2^23 × 1e6 =", round(10 / 2**23 * 1e6, 5), "→ ±10 V 24-bit ADC의 1 LSB(µV) 가설 (추정)")
vs.round(8)

normal: 전류/1.19209 정수 비율 0.0019, 정수 몫 범위 -228~229
outlier: 전류/1.19209 정수 비율 0.9967, 정수 몫 범위 -335~452
1.19209 ≈ 10 V / 2^23 × 1e6 = 1.19209 → ±10 V 24-bit ADC의 1 LSB(µV) 가설 (추정)


,src,channel,n,n_unique,min_step,p1_step,float32_roundtrip_maxerr,vmin,vmax
0,normal,AI0_Vibration,20000,19124,0.000001,0.000001,1.000000e-08,-0.314570,0.351699
1,normal,AI1_Vibration,20000,19460,0.000001,0.000001,1.000000e-08,-0.366498,0.399782
2,normal,AI2_Current,20000,19934,0.000110,0.000240,1.522000e-05,-271.568330,273.235000
3,outlier,AI0_Vibration,600,594,0.000011,0.000032,6.000000e-08,-1.502735,1.803204
4,outlier,AI1_Vibration,600,597,0.000012,0.000023,3.000000e-08,-0.751253,0.509321
5,outlier,AI2_Current,600,340,1.192090,1.192090,1.483000e-05,-399.351170,538.826050


In [14]:
# DC 제거·6자리 반올림 후 샘플 분포 KS 검정: 정상 저부하 vs 이상 조용한 세그먼트
def _samples(uids, ch):
    d = raw[raw["seg_uid"].isin(uids)]
    x = d[ch].round(6)
    return (x - d.groupby("seg_uid")[ch].transform("mean")).to_numpy()
ks_rows = []
for ch in pp.VIB_CHANNELS:
    a = _samples(nf.index[nf["group"] == "normal_state1(저부하)"], ch)
    b = _samples(nf.index[nf["group"] == "outlier_quiet(3·19·20)"], ch)
    ks = stats.ks_2samp(a, b)
    ks_rows.append({"channel": ch, "n_normal_state1": len(a), "n_outlier_quiet": len(b), "sd_normal": round(a.std(), 4),
                    "sd_outlier": round(b.std(), 4), "ks_D": round(ks.statistic, 4), "p_value": ks.pvalue})
ks_tab = pd.DataFrame(ks_rows)
ks_tab.to_csv(RES / "noise_floor_ks.csv", index=False, encoding="utf-8-sig")
ks_tab

,channel,n_normal_state1,n_outlier_quiet,sd_normal,sd_outlier,ks_D,p_value
0,AI0_Vibration,11146,56,0.0569,0.0265,0.2169,0.008576
1,AI1_Vibration,11146,56,0.0610,0.0560,0.1625,0.092479


### A-2 결론 — 게이트 **G2-b**: 스케일 동일성을 확인하지 못했다. 게인 차이 가능성을 한계로 두고 A-3에 세그먼트 내 z-score 변형을 추가한다

- 잡음 바닥(6자리 반올림 후, diff-MAD) 중앙값: 정상 저부하(state1, 304세그먼트) AI0 0.0614 / AI1 0.0600, 이상 조용한 세그먼트 3·19·20은 AI0 0.0236 / AI1 0.0201 → **비율 0.385 / 0.334**로 판정 범위(0.5~2배) 밖이다. 이상 파일의 "조용한" 세그먼트는 정상 저부하보다도 잡음 바닥이 낮다.
- 그러나 이 추정치는 스케일 판정에 쓰기 어렵다. 10 Hz 샘플링에서는 진동이 광대역으로 접혀 들어와 샘플 간 차분이 곧 신호 진폭이므로, 정상 안에서 잡음 바닥과 AC RMS의 Spearman 상관이 0.66(AI0)·0.91(AI1)에 달한다. 즉 잡음 바닥 ≈ 진폭의 다른 표현이고, 이상 3·19·20의 낮은 값은 "게인이 낮다"와 "그 순간 진동이 실제로 작았다"를 구분하지 못한다. 이상 나머지 14세그먼트의 잡음 바닥(AI0 0.0711)은 정상 고부하(0.0749)와 같은 수준이라 게인이 체계적으로 낮다고 보기도 어렵다.
- 값 해상도: 정상 진동은 소수 6자리 반올림(최소 간격 1e-6, 고유값 19,124/20,000), 이상 진동은 반올림 없는 연속값(최소 간격 약 1.1e-5, 고유값 594/600). 둘 다 float32 왕복 오차가 1e-7 미만(정상 1.0e-8 / 이상 6.0e-8)이라 **원 데이터는 모두 float32 정밀도**이고 정상 파일만 내보내기 때 6자리로 잘린 것으로 보인다(추정). 전류는 정상이 연속값(최소 간격 0.00011), 이상은 1.19209 격자 정수배 99.67%·정수 몫 −335~452. 1.19209 ≈ 10 V/2²³×10⁶이므로 이상 파일 전류는 "±10 V 24-bit ADC 코드 × 1 LSB(µV)"로 저장된 원시값이라는 가설이 성립하며(추정), 정상 파일은 추가 스케일링·필터를 거친 값으로 보인다. 어느 쪽이든 **전류 채널은 두 파일의 저장 경로가 다르다**는 결론은 강화된다.
- KS 검정(DC 제거·6자리 반올림 후 샘플 분포, 정상 저부하 11,146 vs 이상 조용 56): AI0 D 0.217 (p 0.009), AI1 D 0.163 (p 0.09). 표본 56개로는 분포 차이의 원인(게인 vs 물리)을 말할 수 없다.

**한계로 기록.** "이상 파일 진동 RMS 최대 6배"는 물리 신호로 서술하되, 게인 차이 가능성을 배제하지 못했음을 보고서 1번 절 한계 표에 적는다. **모델 단계 반영.** A-3에서 진폭 피처를 (i) 원시 스케일, (ii) 세그먼트 내 z-score의 두 변형으로 비교한다. z-score 변형은 스케일 차이에 불변이지만 진폭 정보 자체를 버리므로, 두 변형의 AUC 차이가 곧 "스케일 정보에 얼마나 기대고 있는가"의 척도가 된다.


## A-3. 형식 누수 차단 증명 — 전처리 후에는 "형식만으로" 파일을 구분할 수 없는가

**목적.** "누수가 있다"(02 §2)에서 끝내지 않고, `preprocess.unify_format` + `remove_dc` 뒤에는 표기·저장 형식만으로 정상/이상 파일을 구분할 수 없음을 실험으로 보인다. 같은 분할로 진폭 피처의 AUC를 재면 "전처리 후에도 남는 진짜 신호"의 크기, 즉 2번 모델의 출발 성능이 나온다.

**설계.** 세그먼트 620개(정상 599 / 이상 21)를 `split.fold_assignment`로 계층화 5-fold(이상 4·4·4·4·5) 분할. 피처군마다 표준화+로지스틱 회귀(`signal_checks.cv_auc_logistic`, class_weight=balanced)를 fold별로 학습해 out-of-fold AUC를 본다.
- **형식 피처**(`preprocess.format_features`): 소수 자릿수 최빈값·평균, |DC|, 고유값 비율, 전류 1.19209 격자 정수배 비율 — 물리 신호가 아닌 표기 정보만. 전처리 **전 / 후** × 채널(진동 / 전류 / 전체).
- **진폭 피처**(전처리 후, 세그먼트 평균 제거된 값): 채널별 RMS·피크·첨도·crest factor·왜도. 두 변형 — (i) 원시 스케일, (ii) 세그먼트 내 z-score 후 계산(= 스케일 불변 형상 피처; RMS는 1로 고정되고 첨도·crest·왜도만 정보를 가짐). A-2 G2-b의 "게인 차이 가능성"에 대한 민감도.
- **전류 사인 잔차 피처**(A-1 부산물: 잔차 RMS·R²·|f−0.6|) — 별도 군.

**판정 규칙(게이트 G3).** 전처리 후 형식 피처 AUC ≤ 0.6이면 채널별 "차단 성공"(G3-a). 0.6 초과 채널은 `unify_format`에 1회만 추가 조치를 시도하고, 그래도 초과면 모델 입력에서 제외한다(G3-b).

In [15]:
y = seg.set_index("seg_uid")["label"].astype(int)
fold = sp.fold_assignment(seg, n_splits=5, seed=SEED)
print("fold별 (세그먼트 수, 이상 수):", [(int((fold == k).sum()), int(((fold == k) & (y.reindex(fold.index) == 1)).sum())) for k in range(5)])

def amp_features(d, zscore=False):
    """세그먼트별 진폭·형상 피처. zscore=True면 세그먼트 내 표준화 후 계산(스케일 불변 변형)."""
    rows = []
    for uid, g in d.groupby("seg_uid", sort=False):
        r = {"seg_uid": uid}
        for ch in pp.SENSORS:
            x = g[ch].to_numpy(float); x = x - x.mean()
            if zscore and x.std() > 0:
                x = x / x.std()
            rms = float(np.sqrt(np.mean(x ** 2)))
            r[f"{ch}_rms"] = rms; r[f"{ch}_peak"] = float(np.abs(x).max())
            r[f"{ch}_kurt"] = float(stats.kurtosis(x)) if len(x) > 3 else 0.0
            r[f"{ch}_crest"] = float(np.abs(x).max() / rms) if rms > 0 else 0.0
            r[f"{ch}_skew"] = float(stats.skew(x)) if len(x) > 2 else 0.0
        rows.append(r)
    return pd.DataFrame(rows).set_index("seg_uid")

pre = pp.preprocess(raw)                      # unify_format(진동 6자리, 전류 1.19209 격자 + 1자리) → remove_dc(mean)
ff_before, ff_after = pp.format_features(raw), pp.format_features(pre)
amp_raw, amp_shape = amp_features(pre), amp_features(pre, zscore=True)
sine = cur.set_index("seg_uid")[["resid_rms", "r2"]].assign(f_dev=(cur.set_index("seg_uid")["f_hz"] - 0.6).abs()).rename(columns=lambda c: f"cur_fit_{c}")
print(ff_before.shape, ff_after.shape, amp_raw.shape, sine.shape)

fold별 (세그먼트 수, 이상 수): [(125, 5), (124, 4), (124, 4), (124, 4), (123, 4)]


(620, 13) (620, 13) (620, 15) (620, 3)


In [16]:
VIB, CUR, ALL = pp.VIB_CHANNELS, [pp.CUR_CHANNEL], pp.SENSORS
def run(exp, X, chans, label):
    cols = [c for c in X.columns if any(c.startswith(ch) for ch in chans)]
    r = sc.cv_auc_logistic(X[cols].reindex(y.index), y, fold)
    return {"experiment": exp, "channels": label, **r}
rows = []
for chans, label in [(VIB, "vibration"), (CUR, "current"), (ALL, "all")]:
    rows += [run("format_before", ff_before, chans, label), run("format_after", ff_after, chans, label),
             run("amp_raw_after", amp_raw, chans, label), run("amp_shape_zscore_after", amp_shape, chans, label)]
rows += [run("sine_resid_after", sine, ["cur_fit"], "current"), run("amp_raw+sine_after", amp_raw.join(sine), ALL + ["cur_fit"], "all")]
abl = pd.DataFrame(rows).round(4)
abl.to_csv(RES / "leak_ablation.csv", index=False, encoding="utf-8-sig")
gate3 = abl[abl["experiment"] == "format_after"].set_index("channels")["auc_oof"]
print("G3 판정(형식 피처 전처리 후 AUC ≤ 0.6):", {k: (round(v, 3), "통과" if v <= 0.6 else "초과") for k, v in gate3.items()})
abl

G3 판정(형식 피처 전처리 후 AUC ≤ 0.6): {'vibration': (0.566, '통과'), 'current': (0.597, '통과'), 'all': (0.637, '초과')}


,experiment,channels,auc_oof,auc_fold_mean,n_features
0,format_before,vibration,1.0000,1.0000,8
1,format_after,vibration,0.5655,0.5492,6
2,amp_raw_after,vibration,0.9231,0.9285,10
3,amp_shape_zscore_after,vibration,0.8870,0.8953,10
4,format_before,current,1.0000,1.0000,5
5,format_after,current,0.5967,0.5946,4
6,amp_raw_after,current,0.9224,0.9233,5
7,amp_shape_zscore_after,current,0.8769,0.8814,5
8,format_before,all,1.0000,1.0000,13
9,format_after,all,0.6366,0.6426,10


In [17]:
# 전처리 후에도 남는 형식 피처의 단독 AUC (어느 채널에 무엇이 남는지)
def _sym_auc(s):
    a = roc_auc_score(y, s.reindex(y.index).fillna(0)); return max(a, 1 - a)
resid = pd.DataFrame({"auc_before": ff_before.apply(_sym_auc), "auc_after": ff_after.apply(_sym_auc)}).round(3)
resid.to_csv(RES / "leak_ablation_single_feature.csv", encoding="utf-8-sig")
resid

,auc_before,auc_after
AI0_Vibration_ndec_mode,1.000,0.525
AI0_Vibration_ndec_mean,1.000,0.540
AI0_Vibration_dc_abs,0.944,0.500
AI0_Vibration_uniq_ratio,0.517,0.517
AI1_Vibration_ndec_mode,1.000,0.560
AI1_Vibration_ndec_mean,1.000,0.522
AI1_Vibration_dc_abs,0.920,0.500
AI1_Vibration_uniq_ratio,0.520,0.520
AI2_Current_ndec_mode,0.539,0.527
AI2_Current_ndec_mean,0.783,0.560


In [18]:
piv = abl[abl["experiment"] != "amp_raw+sine_after"].pivot(index="channels", columns="experiment", values="auc_oof").reindex(["vibration", "current", "all"])
order = ["format_before", "format_after", "amp_raw_after", "amp_shape_zscore_after", "sine_resid_after"]
piv = piv[[c for c in order if c in piv.columns]]
fig, ax = plt.subplots(figsize=(10, 4))
w = 0.16; xs = np.arange(len(piv))
colors = {"format_before": "tab:red", "format_after": "salmon", "amp_raw_after": "tab:blue", "amp_shape_zscore_after": "lightsteelblue", "sine_resid_after": "tab:green"}
for i, c in enumerate(piv.columns):
    v = piv[c].values
    ax.bar(xs + (i - len(piv.columns) / 2 + .5) * w, np.nan_to_num(v, nan=0), w, label=c, color=colors.get(c))
    for xx, vv in zip(xs, v):
        if not np.isnan(vv): ax.text(xx + (i - len(piv.columns) / 2 + .5) * w, vv + .01, f"{vv:.2f}", ha="center", fontsize=7)
ax.axhline(0.6, ls="--", c="gray", lw=.8); ax.text(len(piv) - .5, .61, "G3 기준 0.6", fontsize=8, ha="right", color="gray")
ax.axhline(0.5, ls=":", c="gray", lw=.8)
ax.set_xticks(xs); ax.set_xticklabels(piv.index); ax.set_ylim(0.4, 1.08); ax.set_ylabel("out-of-fold AUC (seg 5-fold)")
ax.set_title("형식 누수 차단 실험: 형식 피처 전/후 vs 진폭 피처 (세그먼트 단위 로지스틱)", fontsize=10); ax.legend(fontsize=7, ncol=3, loc="lower left")
paths.save_fig(FIG, "leak_ablation_bar.png")

saved figures\03_diagnosis_recheck_CHS\leak_ablation_bar.png


### A-3 결론 — 게이트 **G3-a(채널별 통과)**: 전처리 후 형식 피처만으로는 파일을 구분할 수 없다. 남는 분리력은 진폭 피처의 것이다

- 형식 피처 로지스틱 out-of-fold AUC(세그먼트 5-fold): **전처리 전 1.000**(진동·전류·전체 모두) → **후 진동 0.566 / 전류 0.597** → 채널별 기준(≤ 0.6) 통과. 전류는 첫 실행에서 0.657로 초과했고, 원인이 1.19209 격자 값의 float 표기 자릿수가 값 크기에 따라 달라지는 것이어서 `unify_format`에 소수 1자리 반올림(최대 값 변화 0.06, 전류 RMS 80~150 대비 무시 가능)을 1회 추가해 0.597로 내렸다(규칙상 허용된 1회 조치).
- 전체 결합은 0.637로 기준을 조금 넘는다. 단독 AUC 표를 보면 전처리 후 남은 것은 고유값 비율(0.517/0.520/0.517)과 자릿수 평균(0.522~0.560)뿐이며, 이는 "진폭이 클수록 반올림 후 고유값이 많아진다"는 **진폭 프록시**이지 파일 저장 형식의 흔적이 아니다(|DC| 0.944/0.920/0.891 → 0.500, 격자 정수배 비율 1.000 → 0.520). 형식 피처 자체는 모델 입력에 쓰지 않으므로 이 잔여는 모델 성능에 영향이 없다. 추가 전처리 탐색은 규칙(E-6)대로 여기서 멈춘다.
- 전처리 후 **진폭 피처** AUC: 진동 0.923 / 전류 0.922 / 전체 0.997. 스케일 불변 형상 변형(세그먼트 내 z-score, 첨도·crest·왜도만 유효)은 0.887 / 0.877 / 0.919 → 진폭 스케일 정보가 기여하는 몫은 진동 0.04, 전류 0.05, 전체 0.08이다. A-2에서 배제하지 못한 게인 차이가 실제라 해도 형상 피처만으로 0.92가 남으므로, 모델 단계는 원시 스케일을 주로 쓰되 형상 변형을 ablation으로 병행 보고한다.
- 전류 사인 잔차 피처군(잔차 RMS·R²·|f−0.6|, 3개)만으로 0.991(fold 평균 0.996), 진폭 15개 + 사인 3개 결합 0.9999. 이 수치는 **지도 학습(로지스틱)·세그먼트 단위·이상 21개**라는 낙관적 조건이므로 2번 모델의 상한 참고치이지 목표치가 아니다. 실제 모델은 정상 데이터만으로 학습해 윈도우 단위로 평가하며, 날짜 교란(이상 이벤트 1건) 때문에 이 AUC의 일부는 "그날의 차이"일 수 있다.

**모델 단계 반영.** (1) 표준 전처리 = `preprocess.preprocess(df)` (진동 6자리, 전류 1.19209 격자 + 1자리, 세그먼트 평균 제거) — 이 순서·인자를 모든 노트북이 공유한다. (2) 형식 피처는 입력 금지. (3) 피처군 3개를 구분해 보고: 진폭(원시), 형상(z-score), 전류 사인 잔차. (4) 이 절의 out-of-fold AUC를 `results/03_diagnosis_recheck_CHS/leak_ablation.csv`로 남겨 모델 비교표의 "지도 상한" 참고 행으로 쓴다.
